## 00 — Loading and Inspecting the Railroad Dataset

Before we can simplify, filter, or display the railroad data — we need to understand what we are holding.

This notebook answers four questions:

1. What is the top-level structure of the file?
2. What geometry type do the features use?
3. What properties does each feature carry?
4. What does one feature actually look like?

The file we are working with is `ne_10m_railroads.geojson` — a 40 MB world railroad dataset from Natural Earth.

## Loading the File

The railroad GeoJSON lives two levels above this notebook in the `data/` folder.

We use `pathlib` to build the path relative to this notebook's location, then Python's built-in `json` module to load it.

In [8]:
import json
from pathlib import Path

data_path = Path("../../data/ne_10m_railroads.geojson")

with open(data_path) as f:
    railroads = json.load(f)

print("Loaded successfully")
print("Top-level keys:", list(railroads.keys()))

Loaded successfully
Top-level keys: ['type', 'name', 'crs', 'features', 'bbox']


## Top-Level Structure

A GeoJSON file is a `FeatureCollection` — a dict with a `type` field and a `features` list.

Let's confirm that and count the features.

In [9]:
print("Type:", railroads["type"])
print("Number of features:", len(railroads["features"]))

Type: FeatureCollection
Number of features: 25413


Over 25,000 railroad segments — each one a separate feature in the collection.

## Inspecting a Single Feature

Every feature in a `FeatureCollection` has the same three keys:
- `type` — always `"Feature"`
- `properties` — a dict of metadata about that feature
- `geometry` — the spatial data (type + coordinates)

Some features also carry a `bbox` field — a precomputed bounding box. This one does.

In [10]:
feature = railroads["features"][0]

print("Feature keys:", list(feature.keys()))
print("Feature type:", feature["type"])
print("Geometry type:", feature["geometry"]["type"])
print("Bbox:", feature["bbox"])

Feature keys: ['type', 'properties', 'bbox', 'geometry']
Feature type: Feature
Geometry type: LineString
Bbox: [30.730275, 69.448054, 30.782502, 69.461111]


## The Geometry

Each feature's geometry is a `LineString` — an ordered list of `[longitude, latitude]` coordinate pairs.

Note the order: **longitude first, latitude second**. This is the GeoJSON convention and it is the opposite of what you might expect from (lat, lon) notation.

In [11]:
coords = feature["geometry"]["coordinates"]

print("Number of coordinate pairs in this feature:", len(coords))
print("First coordinate [lon, lat]:", coords[0])
print("Last coordinate  [lon, lat]:", coords[-1])

Number of coordinate pairs in this feature: 6
First coordinate [lon, lat]: [30.782502, 69.461111]
Last coordinate  [lon, lat]: [30.730275, 69.448054]


## The Properties

Properties are the attribute data attached to each feature — the non-spatial fields.

Let's print all of them for the first feature to see what information we have.

In [12]:
props = feature["properties"]

for key, value in props.items():
    print(f"  {key}: {value}")

  rwdb_rr_id: 1
  mult_track: 1
  electric: 1
  other_code: 1
  category: 1
  disp_scale: 1:3m
  add: 0
  featurecla: Railroad
  scalerank: 10
  natlscale: 1
  part: ne_global_not_north_america
  continent: Europe


Some useful properties to know:

| Property | Meaning |
|---|---|
| `featurecla` | Feature class — e.g. `"Railroad"` |
| `scalerank` | Natural Earth scale rank — lower = more important |
| `natlscale` | Intended display scale (e.g. `250` = 1:250,000) |
| `mult_track` | Whether the line has multiple tracks |
| `electric` | Whether the line is electrified |
| `category` | Broad category of the railroad line |

## Checking Geometry Types Across All Features

We assumed all features are `LineString` — let's verify. A real dataset can have mixed geometry types.

In [13]:
geometry_types = set()

for feature in railroads["features"]:
    geometry_types.add(feature["geometry"]["type"])

print("Geometry types found:", geometry_types)

Geometry types found: {'LineString'}


Only `LineString` — no mixed types to handle. This simplifies our work.

If we had `MultiLineString` features, each one would contain a list of lists of coordinates instead of a flat list. We'd need to handle them separately.

## Exercise A

Print a **sorted list of unique values** for the `category` property across all features.

How many distinct categories exist?

In [14]:
categories = sorted({feature.get("properties", {}).get("category", "<missing>") for feature in features})
print("Distinct categories:", len(categories))
for category in categories:
    print(category)


NameError: name 'features' is not defined

## Exercise B

The `scalerank` property is an integer indicating importance — **lower values = more important** features.

Count how many features exist at each `scalerank` value. Print the results sorted by scalerank.

In [ ]:
from collections import Counter
scalerank_counts = Counter(feature.get("properties", {}).get("scalerank") for feature in features)
for scalerank, count in sorted(scalerank_counts.items(), key=lambda item: (item[0] is None, item[0])):
    print(f"scalerank={scalerank}: {count:,}")


## Check Your Understanding

The most common `natlscale` is the mode of the `natlscale` property values. I would verify it with:

```python
from collections import Counter
Counter(f["properties"].get("natlscale") for f in features).most_common(1)
```

---


## Next

In [01 — Measuring the Problem](./01-Measuring_the_Problem.ipynb), we count the total number of coordinate points, measure load time, and quantify exactly how much data we are dealing with.